# Earnings Call Sentiment Analysis — Full Pipeline
**Course:** CS-4267-02 Deep Learning  
**Authors:** Simon Gou, Davis Tameh-Nfon

End-to-end pipeline:
1. GPU check & environment setup
2. Load transcript data
3. Compute CAR labels
4. FinBERT inference (with checkpointing)
5. Visualize sentiment distributions
6. Train FinBERT prediction head (4 variants)
7. Train baseline models (TF-IDF + NB / LR)
8. Full model comparison
9. Financial metrics

## 1. GPU Check

In [ ]:
import torch

print(f"GPU available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name      : {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory    : {mem_gb:.1f} GB")
else:
    print("WARNING: No GPU detected. Inference will be very slow on CPU.")

## 2. Environment Setup

Mount Google Drive to persist output CSVs across sessions, then clone the repo and install dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = "/content/drive/MyDrive/earnings-sentiment/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Clone repo (skip if already cloned)
import os

REPO_DIR = "/content/Earnings-Call-Sentiment-Analysis-FinBERT-vs.-Naive-Bayes-LR-TF-IDF"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/simongydvandy/Earnings-Call-Sentiment-Analysis-FinBERT-vs.-Naive-Bayes-LR-TF-IDF.git {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes.")
    !git -C {REPO_DIR} pull

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Repo ready.")

In [ ]:
# Install dependencies (uv not available in Colab — use pip directly)
!pip install -q transformers datasets yfinance scikit-learn scipy
print("Dependencies installed.")

## 3. Load Transcript Data

In [ ]:
from shared.load_data import load_transcripts

df = load_transcripts()
df.head(3)

## 4. Compute CAR Labels

Set `SAMPLE_N = None` to run on the full dataset. Use a small number (e.g. 200) for local testing.

This step downloads stock prices from yfinance — expect ~1-3 seconds per row.

In [ ]:
from tqdm.notebook import tqdm
import pandas as pd
from shared.compute_car import _fetch_abnormal_returns, get_train_test_split
import numpy as np

SAMPLE_N = None  # Set to 200 for local dev, None for full run

subset = df.head(SAMPLE_N).copy() if SAMPLE_N else df.copy()
subset = subset.reset_index(drop=True)

car2_list, car5_list = [], []

for _, row in tqdm(subset.iterrows(), total=len(subset), desc="Computing CARs"):
    try:
        abnormal = _fetch_abnormal_returns(row["ticker"], str(row["date"]), window=5)
        if abnormal is None:
            car2_list.append(None)
            car5_list.append(None)
        else:
            car2_list.append(float(abnormal[:2].sum()))
            car5_list.append(float(abnormal[:5].sum()))
    except Exception:
        car2_list.append(None)
        car5_list.append(None)

subset["CAR_2"] = car2_list
subset["CAR_5"] = car5_list
subset = subset.dropna(subset=["CAR_2", "CAR_5"]).reset_index(drop=True)
subset["label_2"] = (subset["CAR_2"] > 0).astype(int)
subset["label_5"] = (subset["CAR_5"] > 0).astype(int)

print(f"\nRows after dropping failures: {len(subset)}")
print(f"label_2 balance: {subset['label_2'].value_counts().to_dict()}")
print(f"label_5 balance: {subset['label_5'].value_counts().to_dict()}")

## 5. FinBERT Inference

Saves to Google Drive every 100 rows. If the session disconnects, rerun this cell — it will resume from the last checkpoint automatically.

In [ ]:
import os, sys
sys.path.insert(0, REPO_DIR)

from finbert.inference import run_pipeline

OUTPUT_PATH     = os.path.join(OUTPUT_DIR, "transcripts_with_sentiment.csv")
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "checkpoint_sentiment.csv")

enriched_df = run_pipeline(
    subset,
    output_path=OUTPUT_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    checkpoint_every=100,
    batch_size=32,
)

## 6. Visualize Sentiment Distributions

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("FinBERT Sentiment Score Distributions", fontsize=14)

for ax, col, color in zip(axes,
                          ["sent_positive", "sent_negative", "sent_neutral"],
                          ["steelblue", "tomato", "gray"]):
    ax.hist(enriched_df[col], bins=40, color=color, edgecolor="white", alpha=0.85)
    ax.set_title(col.replace("sent_", "").capitalize())
    ax.set_xlabel("Probability")
    ax.set_ylabel("Count")
    ax.axvline(enriched_df[col].mean(), color="black", linestyle="--", linewidth=1, label=f"mean={enriched_df[col].mean():.2f}")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "sentiment_distributions.png"), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle("Label Balance (CAR > 0)", fontsize=14)

for ax, col, title in zip(axes, ["label_2", "label_5"], ["2-Day CAR", "5-Day CAR"]):
    counts = enriched_df[col].value_counts().sort_index()
    ax.bar(["Negative (0)", "Positive (1)"], counts.values, color=["tomato", "steelblue"], edgecolor="white")
    ax.set_title(title)
    ax.set_ylabel("Count")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha="center", fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "label_balance.png"), dpi=150)
plt.show()

## 7. Train Prediction Head

4 variants: 2 modes (sentiment / embedding) × 2 label columns (label_2 / label_5)

In [ ]:
from shared.compute_car import get_train_test_split
from finbert.inference import train_prediction_head

# Use label_2 split for index arrays — indices are the same regardless of label_col
train_idx, test_idx, _, _ = get_train_test_split(enriched_df, label_col="label_2")

results = []

for mode in ["sentiment", "embedding"]:
    for label_col in ["label_2", "label_5"]:
        print(f"\nTraining: mode={mode}, label={label_col}")
        metrics = train_prediction_head(enriched_df, train_idx, test_idx, label_col=label_col, mode=mode)
        results.append(metrics)

print("\nAll variants trained.")

## 8. Baseline Models

TF-IDF + Naive Bayes and TF-IDF + Logistic Regression, trained end-to-end directly on CAR labels.
Runs both models for both label columns automatically.

In [ ]:
from baselines.tfidf_models import run_all_label_variants

print("Training baseline models...")
baseline_results = run_all_label_variants(enriched_df)
print(f"Done — {len(baseline_results)} baseline results (2 models × 2 label cols)")

## 9. Full Model Comparison

All 8 results side by side: 4 FinBERT variants + 2 Naive Bayes + 2 Logistic Regression.

In [ ]:
from evaluation.compare_models import compare_models

all_results = results + baseline_results
comparison = compare_models(enriched_df, all_results)

# Display full table with best values highlighted
display(
    comparison[
        ["model", "label_col", "accuracy", "precision", "recall", "f1", "roc_auc",
         "avg_car_positive", "avg_car_negative", "ttest_p_value", "sharpe_ratio"]
    ].style
    .format({
        "accuracy": "{:.4f}", "precision": "{:.4f}", "recall": "{:.4f}",
        "f1": "{:.4f}", "roc_auc": "{:.4f}",
        "avg_car_positive": "{:.4f}", "avg_car_negative": "{:.4f}",
        "ttest_p_value": "{:.4f}", "sharpe_ratio": "{:.3f}",
    })
    .highlight_max(subset=["accuracy", "f1", "roc_auc", "sharpe_ratio"], color="lightgreen")
    .highlight_min(subset=["ttest_p_value"], color="lightyellow")
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

for label_col in ["label_2", "label_5"]:
    subset = comparison[comparison["label_col"] == label_col].reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(12, 4))

    colors = ["steelblue", "royalblue", "tomato", "firebrick",
              "seagreen", "mediumseagreen", "darkorange", "sandybrown"]
    bars = ax.bar(subset["model"], subset["roc_auc"],
                  color=colors[:len(subset)], edgecolor="white", alpha=0.85)

    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, label="Random baseline")
    ax.set_ylim(0.4, 0.85)
    ax.set_ylabel("ROC-AUC")
    ax.set_title(f"All Models — ROC-AUC ({label_col.replace('_', ' ').title()})", fontsize=13)
    ax.legend()
    plt.xticks(rotation=15, ha="right")

    for bar, val in zip(bars, subset["roc_auc"]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f"{val:.4f}", ha="center", fontsize=9)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"all_models_roc_auc_{label_col}.png"), dpi=150)
    plt.show()

In [ ]:
for label_col in ["label_2", "label_5"]:
    subset = comparison[comparison["label_col"] == label_col].reset_index(drop=True)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(f"ML Metrics by Model ({label_col.replace('_', ' ').title()})", fontsize=13)

    for ax, metric in zip(axes, ["f1", "precision", "recall"]):
        colors = ["steelblue" if "FinBERT" in m else "tomato" for m in subset["model"]]
        ax.bar(subset["model"], subset[metric], color=colors, edgecolor="white", alpha=0.85)
        ax.set_title(metric.upper())
        ax.set_ylabel(metric.capitalize())
        ax.set_ylim(0, 1)
        plt.setp(ax.get_xticklabels(), rotation=15, ha="right", fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"all_models_ml_metrics_{label_col}.png"), dpi=150)
    plt.show()

# Legend note
print("Blue bars = FinBERT variants | Red bars = TF-IDF baselines")

In [ ]:
print("=" * 60)
print("FINANCIAL METRICS SUMMARY")
print("=" * 60)

fin_cols = ["model", "label_col", "avg_car_positive", "avg_car_negative",
            "ttest_p_value", "sharpe_ratio"]

for label_col in ["label_2", "label_5"]:
    print(f"\n--- {label_col.replace('_', ' ').upper()} ---")
    subset = comparison[comparison["label_col"] == label_col][fin_cols].reset_index(drop=True)
    for _, row in subset.iterrows():
        sig = "*** SIGNIFICANT" if row["ttest_p_value"] < 0.05 else "not significant"
        print(
            f"  {row['model']:<35}"
            f"  avg_CAR_pos={row['avg_car_positive']:+.4f}"
            f"  avg_CAR_neg={row['avg_car_negative']:+.4f}"
            f"  p={row['ttest_p_value']:.4f} ({sig})"
            f"  Sharpe={row['sharpe_ratio']:.3f}"
        )

print("\n" + "=" * 60)
print("Best model by ROC-AUC (label_2):")
best = comparison[comparison["label_col"] == "label_2"].sort_values("roc_auc", ascending=False).iloc[0]
print(f"  {best['model']}  ROC-AUC={best['roc_auc']:.4f}  Sharpe={best['sharpe_ratio']:.3f}")